# Conformal Backfill Retrieval — Section 2.3 GPU runner

This notebook generates frozen **modality-aware** cosine top-$L$ retrieval logs for all four 200-question datasets, builds the false-match reference banks, and downloads one result ZIP. No manual file upload is required.

This step uses **Jina CLIP v2 embeddings, not Qwen generation**, so vLLM is neither required nor useful. An L4 24 GB GPU should run it; an A100 is faster. CUDA OOM during encoding automatically halves the affected batch and retries.

> The bundled 800 questions are smoke/development data. The resulting artifact is deliberately marked `is_paper_ready=false` when a reference bank has fewer than 1,000 false calibration scores. Do not report a formal conformal guarantee from this small bundle.

In [ ]:
# Colab setup — all Python packages are installed with uv.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/NguyenKhanh2603/Uncertainty-Aware-Iterative-RAG.git"
CODE_COMMIT = "a3c74271deffc339ea42053580a1d71601164b11"
PROJECT_DIR = Path("/content/uncertainty-aware-rag")

uv_executable = shutil.which("uv")
if uv_executable is None:
    installer = Path("/tmp/install-uv.sh")
    subprocess.check_call(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", str(installer)])
    subprocess.check_call(["sh", str(installer)])
    uv_executable = str(Path.home() / ".local" / "bin" / "uv")
if not Path(uv_executable).is_file():
    raise RuntimeError(f"uv installation failed: {uv_executable}")

packages = [
    "transformers==4.46.3",
    "huggingface_hub>=0.26.0",
    "einops>=0.8.0",
    "timm>=1.0.0",
    "Pillow>=10.0.0",
    "numpy>=1.26.0",
    "tqdm>=4.66.0",
    "safetensors>=0.4.0",
]
subprocess.check_call([uv_executable, "pip", "install", "--system", *packages])

if (PROJECT_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "fetch", "origin", CODE_COMMIT])
elif PROJECT_DIR.exists():
    raise RuntimeError(f"{PROJECT_DIR} exists but is not a git checkout; remove or rename it")
else:
    subprocess.check_call(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(PROJECT_DIR)])
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "fetch", "origin", CODE_COMMIT])
subprocess.check_call(["git", "-C", str(PROJECT_DIR), "checkout", "--detach", CODE_COMMIT])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun this cell")
gpu = torch.cuda.get_device_properties(0)
import transformers
import torchvision
print(f"Ready: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")
print(f"Python={sys.version.split()[0]} torch={torch.__version__} torchvision={torchvision.__version__} transformers={transformers.__version__}")
print(f"Code: {REPO_URL} commit={CODE_COMMIT}")

## Configuration

The defaults run 800 questions: MMQA, WebQA, HotpotQA, and TAT-QA, 200 each. The reserve pool keeps at least 10 candidates from every modality available in that dataset and fills the remaining positions by global cosine score. The first smoke bank conditions on `dataset + modality`; query-type and rank stratification should only be enabled after enough calibration data are available.

In [ ]:
DATASETS = ["mmqa", "webqa", "hotpotqa", "tatqa"]
TOP_L = 30
RETRIEVAL_MODE = "modality_aware"
MIN_PER_MODALITY = 10
BANK_CONDITIONING = "dataset,modality"
RANK_BINS = "1-3,4-10,11-30"  # only used if rank_bin is added to BANK_CONDITIONING
MIN_BANK_SIZE = 1000
QUERY_TYPE_MODE = "pooled"
TEXT_BATCH_SIZE = 32       # reduce manually only if repeated batch-size-1 OOM occurs
IMAGE_BATCH_SIZE = 8
QUERY_BATCH_SIZE = 64
CORPUS_BLOCK_SIZE = 16384
OUTPUT_DIR = Path("/content/conformal_backfill_2_3_results")
CACHE_DIR = Path("/content/conformal_backfill_embedding_cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Will process", len(DATASETS) * 200, "questions")

## Generate frozen top-L retrieval logs on GPU

Each dataset gets a JSONL.GZ log and a manifest. Corpus embeddings are cached, so rerunning a completed dataset avoids recomputing them.

In [ ]:
runner = PROJECT_DIR / "scripts" / "generate_conformal_retrieval_log.py"
python_env = os.environ.copy()
python_env["PYTHONPATH"] = str(PROJECT_DIR / "src") + os.pathsep + python_env.get("PYTHONPATH", "")
python_env["PYTHONUNBUFFERED"] = "1"
log_paths = []
for dataset in DATASETS:
    output_path = OUTPUT_DIR / f"{dataset}_top{TOP_L}_retrieval.jsonl.gz"
    command = [
        sys.executable, str(runner),
        "--dataset", dataset,
        "--output", str(output_path),
        "--cache-dir", str(CACHE_DIR),
        "--device", "cuda",
        "--dtype", "float16",
        "--top-l", str(TOP_L),
        "--retrieval-mode", RETRIEVAL_MODE,
        "--min-per-modality", str(MIN_PER_MODALITY),
        "--query-type-mode", QUERY_TYPE_MODE,
        "--text-batch-size", str(TEXT_BATCH_SIZE),
        "--image-batch-size", str(IMAGE_BATCH_SIZE),
        "--query-batch-size", str(QUERY_BATCH_SIZE),
        "--corpus-block-size", str(CORPUS_BLOCK_SIZE),
        "--non-support-label", "false",
        "--data-grade", "smoke",
    ]
    print(f"\n===== {dataset} =====", flush=True)
    process_log = OUTPUT_DIR / f"{dataset}_runner.log"
    with process_log.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command, cwd=PROJECT_DIR, env=python_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_handle.write(line)
        return_code = process.wait()
    if return_code != 0:
        tail = process_log.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]
        raise RuntimeError(
            f"{dataset} retrieval failed with exit code {return_code}. "
            f"Full child-process output is above and in {process_log}.\n"
            + "\n".join(tail)
        )
    log_paths.append(output_path)
print("Finished retrieval logs:", [path.name for path in log_paths])

## Build the combined false-match reference banks

The first test pools ranks within each dataset/modality bank. The 1,000-score gate remains active, but `--allow-small-banks` lets this 800-question smoke run finish while marking underpowered banks explicitly.

In [ ]:
bank_builder = PROJECT_DIR / "scripts" / "prepare_conformal_reference_banks.py"
bank_path = OUTPUT_DIR / "combined_reference_banks.json.gz"
subprocess.check_call([
    sys.executable, str(bank_builder),
    "--input", *[str(path) for path in log_paths],
    "--output", str(bank_path),
    "--rank-bins", RANK_BINS,
    "--conditioning", BANK_CONDITIONING,
    "--min-bank-size", str(MIN_BANK_SIZE),
    "--allow-small-banks",
], cwd=PROJECT_DIR, env=python_env)

import gzip
import json
with gzip.open(bank_path, "rt", encoding="utf-8") as handle:
    banks = json.load(handle)
print(json.dumps({
    "is_paper_ready": banks["is_paper_ready"],
    "top_l": banks["top_l"],
    **banks["summary"],
}, indent=2))
if not banks["is_paper_ready"]:
    print("Expected for the 800Q smoke bundle: collect more calibration queries before making a formal risk-control claim.")

## Package and download results

In [ ]:
archive_base = Path("/content/conformal_backfill_2_3_results")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR))
print(f"Created {archive_path} ({archive_path.stat().st_size / 2**20:.2f} MiB)")
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Not running in Colab; download manually from:", archive_path)